# Stage 2 — Reward Model Training

This notebook trains a **reward model** that scores how good an assistant response is.
It learns from human preference pairs: `chosen` (better) vs `rejected` (worse).

| Item | Detail |
|------|--------|
| **Prerequisite** | SFT LoRA adapter in `Model/sft_model_zip (1)/` (notebook 1 optional) |
| **SFT tokenizer** | Loaded from the SFT adapter folder (keeps vocab aligned) |
| **Base model** | Auto-read from `adapter_config.json` → `Qwen/Qwen2.5-1.5B-Instruct` |
| **Training data** | `reward_train.jsonl` from [chandana5312/rlhf-dataset](https://www.kaggle.com/datasets/chandana5312/rlhf-dataset) |
| **Output** | `REWARD_OUTPUT_DIR` → `/kaggle/working/reward_model` |
| **Next step** | Notebook 4 (`4_ppo-training.ipynb`) loads this reward model for PPO |

---

## What is a reward model?

For each prompt, the model sees a full conversation and outputs a **single score**.
During training it learns:

```
score(prompt + chosen)  >  score(prompt + rejected)
```

The reward model is **separate** from the SFT policy — we only borrow the SFT **tokenizer**.
A fresh classification head (`score.weight`) is trained on top of the base LM.

---

## Kaggle setup

1. **Settings** → Accelerator → **GPU T4** (required for 4-bit training)
2. **Add Input** → `rlhf-dataset` (chandana5312)
3. Keep `USE_KAGGLE_PATHS = True` in the path cell

> You do **not** need to re-run SFT if `Model/sft_model_zip (1)/` already contains a trained adapter.


## 1. Install dependencies

GPU + `bitsandbytes` required for 4-bit QLoRA.


In [2]:
!pip install transformers peft bitsandbytes accelerate trl -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 92.9 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requir

## 2. Import libraries


In [3]:
import torch
import json
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, TaskType
from trl import RewardTrainer, RewardConfig

print("Libraries imported!")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected — 4-bit training will fail on CPU.")


Libraries imported!
GPU: Tesla T4


## 3. Configure paths

Sets `DATA_DIR`, `SFT_MODEL_PATH`, `REWARD_DATA_FILE`, and `BASE_MODEL_NAME`.

On Kaggle, add **chandana5312/rlhf-dataset** via Add Input. Locally, data is read from `Dataset/` and SFT from `Model/sft_model_zip (1)/`.


In [4]:
# --- Path configuration ---
# USE_KAGGLE_PATHS=True uses Kaggle layout when /kaggle/input exists;
# otherwise mirrors repo Dataset/ and Model/ into /kaggle/input paths.

from pathlib import Path
import json
import shutil

USE_KAGGLE_PATHS = True
IS_KAGGLE = Path("/kaggle/input").exists()

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "Notebooks" else NOTEBOOK_DIR
LOCAL_DATASET = REPO_ROOT / "Dataset"

# SFT adapter location varies: local repo, Kaggle input, or uploaded notebook tree
SFT_MODEL_CANDIDATES = [
    NOTEBOOK_DIR / "Model" / "sft_model_zip (1)",
    REPO_ROOT / "Notebooks" / "Model" / "sft_model_zip (1)",
    Path("/kaggle/working/Notebooks/Model/sft_model_zip (1)"),
    Path("/kaggle/input/sft-model"),
    Path("/kaggle/input/datasets/chandana5312/sft-model"),
    Path("/kaggle/working/sft-model"),
]


def find_kaggle_dataset_dir():
    """Find rlhf-dataset mount containing reward_train.jsonl."""
    candidates = [
        Path("/kaggle/input/rlhf-dataset"),
        Path("/kaggle/input/datasets/chandana5312/rlhf-dataset"),
    ]
    for path in candidates:
        if (path / "reward_train.jsonl").exists():
            return path
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for path in kaggle_input.rglob("reward_train.jsonl"):
            return path.parent
    return candidates[-1]


def is_sft_adapter_dir(path):
    """True if folder contains a PEFT LoRA adapter (not just a training checkpoint)."""
    return (path / "adapter_config.json").exists() and (
        (path / "adapter_model.safetensors").exists()
        or (path / "adapter_model.bin").exists()
    )


def find_sft_model_dir():
    """Find SFT LoRA adapter under /kaggle/input or local Model/."""
    # 1) On Kaggle: search ALL mounted inputs first (any dataset slug / nested zip folder)
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        matches = []
        for cfg in kaggle_input.rglob("adapter_config.json"):
            parent = cfg.parent
            if is_sft_adapter_dir(parent):
                matches.append(parent)
        if matches:
            # Prefer root-level mounts (e.g. /kaggle/input/sft-model) over checkpoint-* subfolders
            matches.sort(key=lambda p: (len(p.parts), "checkpoint" in p.name))
            return matches[0]

    # 2) Known local / repo paths
    for path in SFT_MODEL_CANDIDATES:
        if is_sft_adapter_dir(path):
            return path

    return None


def print_kaggle_input_tree():
    """Debug helper: show what is mounted under /kaggle/input."""
    kaggle_input = Path("/kaggle/input")
    if not kaggle_input.exists():
        print("No /kaggle/input on this machine.")
        return
    print("Mounted under /kaggle/input/:")
    for p in sorted(kaggle_input.iterdir()):
        print(f"  {p.name}/")
        if p.is_dir():
            for child in sorted(p.iterdir())[:8]:
                suffix = "/" if child.is_dir() else ""
                print(f"    {child.name}{suffix}")
            extra = len(list(p.iterdir())) - 8
            if extra > 0:
                print(f"    ... (+{extra} more)")


def setup_local_data_mirror():
    data_dir = Path("/kaggle/input/rlhf-dataset")
    if (data_dir / "reward_train.jsonl").exists():
        return data_dir
    if LOCAL_DATASET.exists():
        data_dir.parent.mkdir(parents=True, exist_ok=True)
        if not data_dir.exists():
            try:
                data_dir.symlink_to(LOCAL_DATASET.resolve(), target_is_directory=True)
            except OSError:
                shutil.copytree(LOCAL_DATASET, data_dir)
    return data_dir


if USE_KAGGLE_PATHS and IS_KAGGLE:
    DATA_DIR = find_kaggle_dataset_dir()
    REWARD_OUTPUT_DIR = Path("/kaggle/working/reward_model")
elif USE_KAGGLE_PATHS:
    DATA_DIR = setup_local_data_mirror()
    REWARD_OUTPUT_DIR = Path("/kaggle/working/reward_model")
else:
    DATA_DIR = LOCAL_DATASET
    REWARD_OUTPUT_DIR = REPO_ROOT / "checkpoints" / "reward_model"

REWARD_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REWARD_DATA_FILE = DATA_DIR / "reward_train.jsonl"

SFT_MODEL_PATH = find_sft_model_dir()

if SFT_MODEL_PATH is None or not is_sft_adapter_dir(SFT_MODEL_PATH):
    print_kaggle_input_tree()
    raise FileNotFoundError(
        "SFT LoRA adapter not found.\n"
        "On Kaggle you must BOTH upload the dataset AND attach it to this notebook:\n"
        "  Notebook sidebar → + Add Input → select your sft-model dataset → Add\n"
        "Then re-run this cell.\n"
        "Zip should contain adapter_config.json + adapter_model.safetensors at the top level\n"
        "or inside one folder (e.g. sft_model_zip (1)/)."
    )

with open(SFT_MODEL_PATH / "adapter_config.json", encoding="utf-8") as f:
    BASE_MODEL_NAME = json.load(f)["base_model_name_or_path"]

print("USE_KAGGLE_PATHS :", USE_KAGGLE_PATHS)
print("IS_KAGGLE        :", IS_KAGGLE)
print("DATA_DIR         :", DATA_DIR.resolve())
print("SFT_MODEL_PATH   :", SFT_MODEL_PATH.resolve())
print("BASE_MODEL_NAME  :", BASE_MODEL_NAME)
print("REWARD_DATA_FILE :", REWARD_DATA_FILE)
print("REWARD_OUTPUT_DIR:", REWARD_OUTPUT_DIR.resolve())

if not REWARD_DATA_FILE.exists():
    raise FileNotFoundError(
        f"Missing {REWARD_DATA_FILE}. On Kaggle: Add Input → chandana5312/rlhf-dataset"
    )
print("  [OK]", REWARD_DATA_FILE)

USE_KAGGLE_PATHS : True
IS_KAGGLE        : True
DATA_DIR         : /kaggle/input/datasets/chandana5312/rlhf-dataset
SFT_MODEL_PATH   : /kaggle/input/datasets/chandana5312/sft-model-dataset
BASE_MODEL_NAME  : Qwen/Qwen2.5-1.5B-Instruct
REWARD_DATA_FILE : /kaggle/input/datasets/chandana5312/rlhf-dataset/reward_train.jsonl
REWARD_OUTPUT_DIR: /kaggle/working/reward_model
  [OK] /kaggle/input/datasets/chandana5312/rlhf-dataset/reward_train.jsonl


## 4. Load tokenizer from SFT checkpoint

We reuse the SFT tokenizer so token IDs match the policy model used in PPO later.


In [5]:
# Tokenizer must match the SFT checkpoint (same vocab as PPO policy later)
tokenizer = AutoTokenizer.from_pretrained(str(SFT_MODEL_PATH))
tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer loaded from {SFT_MODEL_PATH}")


Tokenizer loaded from /kaggle/input/datasets/chandana5312/sft-model-dataset


## 5. Load reward training data

Each row in `reward_train.jsonl` has:

| Column | Meaning |
|--------|--------|
| `prompt` | User message |
| `chosen` | Preferred assistant reply |
| `rejected` | Worse assistant reply |


In [6]:
def load_jsonl(file):
    """Load a JSONL file into a list of dicts."""
    with open(file, encoding="utf-8") as f:
        return [json.loads(line) for line in f]

data = load_jsonl(REWARD_DATA_FILE)
print(f"Loaded {len(data)} samples from {REWARD_DATA_FILE}")
print(f"Columns: {list(data[0].keys())}")
print(f"\nSample:")
print(data[0])


Loaded 1927 samples from /kaggle/input/datasets/chandana5312/rlhf-dataset/reward_train.jsonl
Columns: ['prompt', 'chosen', 'rejected']

Sample:
{'prompt': 'Can you describe why a drought does not end when it rains', 'chosen': "A drought is usually defined as a long period of time with a shortage of water. While rain is necessary to replenish our water supplies, it isn’t a very reliable way of providing water. This can be attributed to a lack of water infrastructure and advanced weather forecasting. Humans have only a limited ability to predict the timing and amount of rainfall.\nOkay. But you didn't answer my question, you simply described a drought Sorry, I'm not sure what you mean. The sentence structure in your question doesn’t fit with what I’m used to. I think you may have been looking for me to explain something else, like why rain doesn’t end a drought? The answer is that it doesn’t end a drought because the amount of water that falls to the ground in a rain event is often not e

## 6. Prepare Hugging Face dataset

`RewardTrainer` tokenizes internally — we only pass raw `prompt` / `chosen` / `rejected` columns.


In [7]:
# RewardTrainer handles tokenization — keep raw text columns only
def format_dataset(example):
    return {
        "prompt": example["prompt"],
        "chosen": example["chosen"],
        "rejected": example["rejected"],
    }

dataset = Dataset.from_list(data)
dataset = dataset.map(format_dataset)

print("Dataset ready!")
print(f"Columns: {dataset.column_names}")
print(f"Size: {len(dataset)} samples")


Map:   0%|          | 0/1927 [00:00<?, ? examples/s]

Dataset ready!
Columns: ['prompt', 'chosen', 'rejected']
Size: 1927 samples


## 7. Train / validation split

90% train, 10% validation (`seed=42` for reproducibility).


In [8]:
split = dataset.train_test_split(test_size=0.1, seed=42)
train_data = split["train"]
val_data = split["test"]

print(f"Train: {len(train_data)} samples")
print(f"Val:   {len(val_data)} samples")


Train: 1734 samples
Val:   193 samples


## 8. Load base model (sequence classification)

Loads `BASE_MODEL_NAME` with a **new** `score` head (`num_labels=1`).

The `score.weight | MISSING` message is expected — that head is trained in this notebook.


In [9]:
# 4-bit quantization — fits Qwen2.5-1.5B reward model on a T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Fresh classification head; SFT LoRA weights are NOT loaded here
print(f"Loading {BASE_MODEL_NAME} for reward head... (2-3 mins)")
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL_NAME,
    num_labels=1,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.pad_token_id = tokenizer.eos_token_id

print("Model loaded!")


Loading Qwen/Qwen2.5-1.5B-Instruct for reward head... (2-3 mins)


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-1.5B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded!


## 9. LoRA configuration

Lightweight adapters on attention layers; keeps VRAM low on a T4.


In [10]:
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"],
)

print("LoRA config ready!")


LoRA config ready!


## 10. Training hyperparameters

| Setting | Value | Notes |
|---------|-------|-------|
| epochs | 1 | One full pass over ~1.9k pairs |
| batch size | 4 | Per GPU |
| grad accumulation | 4 | Effective batch = 16 |
| max_length | 512 | Longer pairs are truncated |
| save_steps | 50 | Checkpoint for resume if session stops |
| resume | auto | Resumes from latest `checkpoint-*` if present |


In [11]:
import os

TRAINING_COMPLETE_MARKER = REWARD_OUTPUT_DIR / ".reward_training_complete"


def find_latest_checkpoint(output_dir):
    """Return path to latest checkpoint-N folder, or None."""
    output_dir = Path(output_dir)
    checkpoints = sorted(
        [p for p in output_dir.glob("checkpoint-*") if p.is_dir()],
        key=lambda p: int(p.name.rsplit("-", 1)[-1]),
    )
    return str(checkpoints[-1]) if checkpoints else None


training_args = RewardConfig(
    output_dir=str(REWARD_OUTPUT_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1.41e-5,
    warmup_steps=50,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=50,              # checkpoint every 50 steps (resume-friendly)
    save_total_limit=3,         # keep last 3 checkpoints
    save_strategy="steps",
    fp16=False,
    bf16=True,
    report_to="none",
    max_length=512,
)

# Required for some trl versions — avoids center_rewards_coefficient error
training_args.center_rewards_coefficient = None

RESUME_CHECKPOINT = find_latest_checkpoint(REWARD_OUTPUT_DIR)

print("Training arguments set!")
print(f"Output dir        : {REWARD_OUTPUT_DIR}")
print(f"Resume checkpoint : {RESUME_CHECKPOINT or 'None (fresh start)'}")
print(f"Training complete : {TRAINING_COMPLETE_MARKER.exists()}")


Training arguments set!
Output dir        : /kaggle/working/reward_model
Resume checkpoint : None (fresh start)
Training complete : False


## 11. Train the reward model

Expect **~1 hour** on a Kaggle T4 for 1 epoch.

Healthy training: `rewards/chosen` trends **up**, `rewards/rejected` trends **down**.

**Resume:** If the session stops, re-run from **§3** onward. Training auto-resumes from the latest `checkpoint-*` in `REWARD_OUTPUT_DIR`. If training already finished, it skips `trainer.train()`.

> **Kaggle tip:** `/kaggle/working/` is cleared when a session fully ends. To resume across sessions, attach your previous notebook **Output** as **Input** (Create → Notebook output), or download `reward_model.zip` before the session closes.


In [12]:
trainer = RewardTrainer(
    model=model,
    args=training_args,
    processing_class=tokenizer,
    train_dataset=train_data,
    eval_dataset=val_data,
    peft_config=peft_config,
)

TRAINING_COMPLETE_MARKER = REWARD_OUTPUT_DIR / ".reward_training_complete"

def _latest_checkpoint(output_dir):
    paths = sorted(
        Path(output_dir).glob("checkpoint-*"),
        key=lambda p: int(p.name.rsplit("-", 1)[-1]),
    )
    return str(paths[-1]) if paths else None

try:
    RESUME_CHECKPOINT
except NameError:
    RESUME_CHECKPOINT = _latest_checkpoint(REWARD_OUTPUT_DIR)

if TRAINING_COMPLETE_MARKER.exists():
    print("Training already complete — skipping trainer.train().")
    print(f"Using saved model in {REWARD_OUTPUT_DIR}")
    model = trainer.model
elif RESUME_CHECKPOINT:
    print(f"Resuming from checkpoint: {RESUME_CHECKPOINT}")
    print("Watch for rewards/chosen going UP and rewards/rejected going DOWN!\n")
    trainer.train(resume_from_checkpoint=RESUME_CHECKPOINT)
    TRAINING_COMPLETE_MARKER.write_text("done", encoding="utf-8")
    print("Reward Model Training complete!")
else:
    print("Starting Reward Model Training (fresh run)...")
    print("Watch for rewards/chosen going UP and rewards/rejected going DOWN!\n")
    trainer.train()
    TRAINING_COMPLETE_MARKER.write_text("done", encoding="utf-8")
    print("Reward Model Training complete!")

model = trainer.model


Adding EOS to train dataset:   0%|          | 0/1734 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1734 [00:00<?, ? examples/s]

Filtering train >512 tokens:   0%|          | 0/1734 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/193 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/193 [00:00<?, ? examples/s]

Filtering eval >512 tokens:   0%|          | 0/193 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Starting Reward Model Training (fresh run)...
Watch for rewards/chosen going UP and rewards/rejected going DOWN!



Step,Training Loss,Validation Loss,Num Tokens,Min Reward,Mean Reward,Max Reward,Accuracy,Margin
50,0.805410,0.769982,265075.000000,-3.719769,-1.535203,0.191905,0.548913,0.077391
100,0.884050,0.763207,523448.000000,-3.715523,-1.558225,0.152386,0.559783,0.084818


Reward Model Training complete!


## 12. Evaluate on validation set

**Preference accuracy** = % of val rows where `score(chosen) > score(rejected)`.

Target: well above 50% (random). 55–70% is reasonable for 1 epoch.


In [13]:
# Preference accuracy: chosen should score higher than rejected
PROMPT_TEMPLATE = "\n\nHuman: {prompt}\n\nAssistant: {response}"

print("Evaluating Reward Model...")
print("=" * 60)

model.eval()
correct = 0
total = min(200, len(val_data))

for i in range(total):
    sample = val_data[i]
    chosen_text = PROMPT_TEMPLATE.format(prompt=sample["prompt"], response=sample["chosen"])
    rejected_text = PROMPT_TEMPLATE.format(prompt=sample["prompt"], response=sample["rejected"])

    chosen_inputs = tokenizer(chosen_text, return_tensors="pt", truncation=True, max_length=512).to("cuda")
    rejected_inputs = tokenizer(rejected_text, return_tensors="pt", truncation=True, max_length=512).to("cuda")

    with torch.no_grad():
        chosen_score = model(**chosen_inputs).logits[0].item()
        rejected_score = model(**rejected_inputs).logits[0].item()

    is_correct = chosen_score > rejected_score
    if is_correct:
        correct += 1

    if i < 5 or i >= total - 3:
        status = "OK" if is_correct else "FAIL"
        print(f"Row {i+1:3d} | chosen: {chosen_score:7.3f} | rejected: {rejected_score:7.3f} | {status}")
    elif i == 5:
        print("...")

accuracy = correct / total * 100
print("=" * 60)
print(f"\nFinal Accuracy: {accuracy:.1f}% ({correct}/{total} correct)")


Evaluating Reward Model...
Row   1 | chosen:  -1.367 | rejected:  -1.500 | OK
Row   2 | chosen:  -2.188 | rejected:  -1.305 | FAIL
Row   3 | chosen:  -2.266 | rejected:  -1.875 | FAIL
Row   4 | chosen:  -1.211 | rejected:  -0.436 | FAIL
Row   5 | chosen:  -1.188 | rejected:  -1.148 | FAIL
...
Row 191 | chosen:  -0.377 | rejected:   0.428 | FAIL
Row 192 | chosen:  -0.157 | rejected:  -0.432 | OK
Row 193 | chosen:  -0.617 | rejected:   0.295 | FAIL

Final Accuracy: 56.5% (109/193 correct)


## 13. Save checkpoint

Saved to `REWARD_OUTPUT_DIR`. Notebook 4 loads this folder as the frozen reward scorer.


In [14]:
trainer.save_model(str(REWARD_OUTPUT_DIR))
tokenizer.save_pretrained(str(REWARD_OUTPUT_DIR))
TRAINING_COMPLETE_MARKER.write_text("done", encoding="utf-8")

print("Reward model saved!")
print(f"Location: {REWARD_OUTPUT_DIR}")
print("Next: run 4_ppo-training.ipynb in the same Kaggle session.")


Reward model saved!
Location: /kaggle/working/reward_model
Next: run 4_ppo-training.ipynb in the same Kaggle session.


## 14. Quick smoke test

Manual check: a sensible answer should score higher than a vague one.


In [15]:
def get_reward_score(prompt, response):
    """Score a single prompt+response pair (higher = better)."""
    text = f"\n\nHuman: {prompt}\n\nAssistant: {response}"
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to("cuda")
    with torch.no_grad():
        return model(**inputs).logits[0].item()

prompt = "How do I stay healthy?"
good_response = "Exercise daily, eat balanced meals, sleep 7-8 hours!"
bad_response = "I don't know just try stuff"

good_score = get_reward_score(prompt, good_response)
bad_score = get_reward_score(prompt, bad_response)

print("Reward Scores:")
print(f"Good response: {good_score:.4f}")
print(f"Bad response:  {bad_score:.4f}")
print("\nReward model working correctly!" if good_score > bad_score else "\nReward model needs more training")


Reward Scores:
Good response: -2.6094
Bad response:  -2.9375

Reward model working correctly!


## 15. Export zip (optional)

Creates `reward_model.zip` for download from Kaggle `/kaggle/working/`.


In [16]:
import shutil

shutil.make_archive(str(REWARD_OUTPUT_DIR), "zip", str(REWARD_OUTPUT_DIR))
print(f"ZIP created: {REWARD_OUTPUT_DIR}.zip")


ZIP created: /kaggle/working/reward_model.zip
